In [1]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/evgenykomarov/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/evgenykomarov/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/evgenykomarov/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/evgenykomarov/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

Starting virtual X frame buffer: Xvfb.


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [2]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env
# nature_dqn_env = atari_wrappers.nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 8  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [3]:
# import tensorflow as torch
# import torch as tf

import torch
import torch.nn as nn
import torch.nn.functional as F

# <YOUR CODE: define your model here>

def init_weights(layer, gain=np.sqrt(2)):
    assert isinstance(layer, (nn.Linear, nn.Conv2d))
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.zeros_(layer.bias)
    return layer

def get_model(env):
    n_actions = env.action_space.n

    # state_dim = env.observation_space.shape
    class A2CModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.n_actions = n_actions

            self.backbone = nn.Sequential(
                # 4 frames 84 x 84
                # Conv2d(W, K, S): W -> (W - K) / S + 1
                init_weights(nn.Conv2d(4, 32, kernel_size=8, stride=4)), # 4 x 84 x 84 -> 32 x 20 x 20
                nn.ReLU(),
                init_weights(nn.Conv2d(32, 64, kernel_size=4, stride=2)), # -> 64 x 9 x 9
                nn.ReLU(),
                init_weights(nn.Conv2d(64, 64, kernel_size=3, stride=1)), # -> 64 x 7 x 7
                nn.ReLU(),
                nn.Flatten(),
                init_weights(nn.Linear(64 * 7 * 7, 512)),
                nn.ReLU(),
            )
            # advantage actor
            self.policy_head = init_weights(nn.Linear(512, n_actions), gain=0.01)
            # critic
            self.value_head = init_weights(nn.Linear(512, 1), gain=1.0)

        def forward(self, x):
            x /= 255.0
            features = self.backbone(x)
            logits = self.policy_head(features) # [batch, n_actions]
            values = self.value_head(features) # [batch, 1]
            return logits, values
    return A2CModel()

model = get_model(env)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [4]:
logits, values = model.forward(torch.FloatTensor(env.reset()[0]))
print(f'logits={logits}')
print(f'values={values}')

logits=tensor([[ 2.3948e-06,  5.6509e-06,  4.2098e-06,  2.8833e-06, -5.7440e-06,
          2.5064e-06],
        [ 2.3948e-06,  5.6509e-06,  4.2098e-06,  2.8833e-06, -5.7440e-06,
          2.5064e-06],
        [ 1.9500e-06,  5.4978e-06,  4.5916e-06,  2.9690e-06, -5.1804e-06,
          2.2143e-06],
        [ 2.3948e-06,  5.6509e-06,  4.2098e-06,  2.8833e-06, -5.7440e-06,
          2.5064e-06],
        [ 1.9500e-06,  5.4978e-06,  4.5916e-06,  2.9690e-06, -5.1804e-06,
          2.2143e-06],
        [ 1.9500e-06,  5.4978e-06,  4.5916e-06,  2.9690e-06, -5.1804e-06,
          2.2143e-06],
        [ 2.3948e-06,  5.6509e-06,  4.2098e-06,  2.8833e-06, -5.7440e-06,
          2.5064e-06],
        [ 1.9500e-06,  5.4978e-06,  4.5916e-06,  2.9690e-06, -5.1804e-06,
          2.2143e-06]], grad_fn=<AddmmBackward0>)
values=tensor([[0.0001],
        [0.0001],
        [0.0002],
        [0.0001],
        [0.0002],
        [0.0002],
        [0.0001],
        [0.0002]], grad_fn=<AddmmBackward0>)


In [5]:
def to_probs(logits):
    """
    Predict action probabilities given logits.
    :param logits: numpy array of shape [batch, n_actions]
    :returns: numpy array of shape [batch, n_actions]
    """
    # convert states, compute logits, use softmax to get probability
    with torch.no_grad():
        probs_tensor = F.softmax(logits, dim=1).numpy()
        log_probs_tensor = F.log_softmax(logits, dim=1).numpy()
    return probs_tensor, log_probs_tensor

def to_probs_tensor(logits):
    """
    Predict action probabilities given logits.
    :param logits: numpy array of shape [batch, n_actions]
    :returns: numpy array of shape [batch, n_actions]
    """
    probs_tensor = F.softmax(logits, dim=1)
    log_probs_tensor = F.log_softmax(logits, dim=1)
    return probs_tensor, log_probs_tensor


In [6]:
probs, log_probs = to_probs_tensor(logits)
torch.tensor([np.random.choice(range(model.n_actions), p=prob.detach().numpy()) for prob in probs], dtype=torch.long)

tensor([3, 3, 5, 4, 0, 2, 0, 5])

In [7]:
class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].
        # states = torch.tensor(states, dtype=torch.float32)
        # actions = torch.tensor(actions, dtype=torch.int64)
        # cumulative_returns = np.array(get_cumulative_rewards(rewards, gamma))
        # cumulative_returns = torch.tensor(cumulative_returns, dtype=torch.float32)
        inputs = torch.tensor(inputs, dtype=torch.float32)
        logits, values = model(inputs)
        probs, log_probs = to_probs_tensor(logits)
        actions = np.array([np.random.choice(range(model.n_actions), p=prob.detach().numpy()) for prob in probs])
        return {
            'actions': actions, # np.array
            'logits': logits, # torch.tensor
            'log_probs': log_probs, # torch.tensor
            'values': values, # torch.tensor
        }


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [8]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [24]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        from pdb import set_trace
        """Compute value targets for a given partial trajectory."""
        # zip(*trajectory)
        # length = len()
        # trajectory['observations']
        # trajectory['rewards']
        # trajectory['resets']
        # trajectory['actions']
        # trajectory['logits']
        # trajectory['log_probs']
        # trajectory['values']

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.
        #
        gamma = self.gamma

        last_state = torch.tensor(trajectory['state']['latest_observation'], dtype=torch.float32) # [batch, ...]
        _, value_last_observation = model(last_state) # [batch, 1]
        T = len(trajectory['rewards'])
        next_value_targets = value_last_observation
        value_targets = [None] * T
        for t in range(T - 1, -1, -1):
            rewards_batch = torch.tensor(trajectory['rewards'][t], dtype=torch.float32).reshape(-1, 1) # [batch, 1]
            value_targets[t] = rewards_batch + gamma * next_value_targets
            resets_batch = torch.tensor(trajectory['resets'][t], dtype=torch.float32).reshape(-1, 1)
            next_value_targets = (1.0 - resets_batch) * value_targets[t]
        trajectory['value_targets'] = value_targets
        # <YOUR CODE>

        # set_trace()

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [32]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        from pdb import set_trace
        # Modify trajectory inplace.
        # <YOUR CODE>
        for key in trajectory:
            if key == 'state':
                continue
            assert isinstance(trajectory[key], list)
            if not trajectory[key]:
                continue
            assert isinstance(trajectory[key][0], (np.ndarray, torch.Tensor))
            if isinstance(trajectory[key][0], np.ndarray):
                trajectory[key] = np.concatenate(trajectory[key])
            elif isinstance(trajectory[key][0], torch.Tensor):
                trajectory[key] = torch.concatenate(trajectory[key])
            else:
                raise Exception

        # set_trace()

In [33]:
model = get_model(env)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


In [34]:
runner.get_next()

TypeError: expected Tensor as element 0 in argument 0, but got numpy.ndarray

In [35]:
from pdb import pm
pm()

> /tmp/ipykernel_60034/4911515.py(13)__call__()
     11                 trajectory[key] = np.concatenate(trajectory[key])
     12                 continue
---> 13             trajectory[key] = torch.concatenate(trajectory[key])
     14 
     15         # set_trace()

ipdb> key
'observations'
ipdb> len(trajectory[key])
5
ipdb> trajectory[key][0]
array([[[[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.30980393, 0.30980393, 0.30980393, ..., 0.30980393,
          0.30980393, 0.30980393],
         [0.30980393, 0.30980393, 0.30980393, ..., 0.30980393,
          0.30980393, 0.30980393],
         [0.30980393, 0.30980393, 0.30980393, ..., 0.30980393,
          0.30980393, 0.30980393]],

        [[0.        , 0.        , 0.        , ...

Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

slides from Levine lecture: https://rail.eecs.berkeley.edu/deeprlcourse-fa17/f17docs/lecture_5_actor_critic_pdf.pdf

In [ ]:
class A2C:
    def __init__(
        self,
        policy,
        optimizer,
        value_loss_coef=0.25,
        entropy_coef=0.01,
        max_grad_norm=0.5,
    ):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        # <YOUR CODE>

        policy = self.policy
        model = policy.model
        gamma = 1.0

        # cast everything into torch tensors
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.int64)
        cumulative_returns = np.array(get_cumulative_rewards(rewards, gamma))
        cumulative_returns = torch.tensor(cumulative_returns, dtype=torch.float32)

        model(state)
        rewards_batch = trajectory['rewards']

        advantage = reward + gamma * value(next_state) - value(state)
        # predict logits, probas and log-probas using an agent.
        logits = model(states)
        probs = nn.functional.softmax(logits, -1)
        log_probs = nn.functional.log_softmax(logits, -1)

        assert all(isinstance(v, torch.Tensor) for v in [logits, probs, log_probs]), \
            "please use compute using torch tensors and don't use predict_probs function"

        # select log-probabilities for chosen actions, log pi(a_i|s_i)
        # log_probs_for_actions = torch.sum(
        #     log_probs * F.one_hot(actions, env.action_space.n), dim=1)
        log_probs_for_actions = log_probs.gather(
            dim=1,
            index=actions[:, None]
        ).squeeze(-1)


        # Compute loss here. Don't forgen entropy regularization with `entropy_coef`
        probs_for_actions = torch.sum(
            probs * F.one_hot(actions, env.action_space.n), dim=1)
        entropy = -entropy_coef * torch.sum(probs_for_actions * log_probs_for_actions)
        loss = -torch.sum(log_probs_for_actions * cumulative_returns) - entropy



    def value_loss(self, trajectory):
        <YOUR CODE>

    def loss(self, trajectory):
        <YOUR CODE>

    def step(self, trajectory):
        <YOUR CODE>

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [ ]:
#if you use TensorboardSummaries
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
a2c = <YOUR CODE: create an instance of the algorithm>

<YOUR CODE: write a training loop>

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.